# Correlation Is Not Causation

## A one-hour data science lesson on confounding and Simpson's paradox

### Central question

A college offers optional tutoring to students. At the end of the semester, the data show that students who attended tutoring had a **lower overall pass rate** than students who did not attend.

Does this mean tutoring harmed students?

This notebook helps learners move beyond surface-level comparisons. Students will examine aggregated data, identify a hidden confounder, discover Simpson's paradox, evaluate competing explanations, use AI as a critic rather than an answer source, and design a stronger study.

> **Core lesson:** A pattern in data may be real without supporting the causal conclusion we want to make.

## Learning objectives

By the end of the session, students should be able to:

- distinguish association from causation;
- explain how confounding can create a misleading comparison;
- calculate overall and subgroup pass rates;
- recognize Simpson's paradox;
- identify variables that should be controlled or stratified;
- evaluate an AI-generated causal claim critically;
- propose a stronger study design;
- communicate uncertainty and limitations clearly;
- transfer the reasoning to another data science context.

## Suggested one-hour schedule

| Time | Activity |
|---|---|
| 0–6 minutes | React to the claim before seeing the full data |
| 6–15 minutes | Build and inspect the dataset |
| 15–23 minutes | Analyze the overall comparison |
| 23–33 minutes | Stratify by prior preparation |
| 33–40 minutes | Explain confounding and Simpson's paradox |
| 40–47 minutes | Use AI as a skeptical reviewer |
| 47–54 minutes | Design a stronger study |
| 54–58 minutes | Transfer the reasoning to another domain |
| 58–60 minutes | Complete an exit reflection |

# Part 1 — Begin With the Claim

The college reports:

> “Students who attended tutoring passed at a lower rate than students who did not attend tutoring.”

Before running any code, answer:

1. What conclusion are you tempted to make?
2. What alternative explanations could produce this pattern?
3. What additional information would you request?
4. What would be dangerous about acting on the claim immediately?

### My initial response

**My first conclusion:**  

**Alternative explanations:**  

**Information still needed:**  

**Risk of acting too quickly:**

# Part 2 — Create the Student-Level Dataset

The fictional dataset contains 400 students. Students are grouped by prior preparation based on a diagnostic assessment:

- **High preparation**
- **Low preparation**

Tutoring was optional, so students were not randomly assigned. Students with lower preparation were much more likely to attend tutoring.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


def make_group(preparation, tutoring, n_students, n_passed, start_id):
    passed = [1] * n_passed + [0] * (n_students - n_passed)
    return pd.DataFrame({
        "student_id": [f"S{i:03d}" for i in range(start_id, start_id + n_students)],
        "preparation": preparation,
        "tutoring": tutoring,
        "passed": passed,
    })

parts = [
    make_group("High", "Yes", 40, 36, 1),
    make_group("High", "No", 160, 136, 41),
    make_group("Low", "Yes", 160, 64, 201),
    make_group("Low", "No", 40, 12, 361),
]

students = pd.concat(parts, ignore_index=True)
students = students.sample(frac=1, random_state=42).reset_index(drop=True)
students.head(10)

## Data-quality check

Before analysis, verify the number of records, missing values, category labels, tutoring participation, preparation groups, and pass outcomes.

In [ ]:
quality_check = {
    "rows": len(students),
    "missing_values": students.isna().sum().to_dict(),
    "tutoring_counts": students["tutoring"].value_counts().to_dict(),
    "preparation_counts": students["preparation"].value_counts().to_dict(),
    "pass_counts": students["passed"].value_counts().to_dict(),
}
quality_check

# Part 3 — Analyze the Overall Comparison

First, compare pass rates for students who attended tutoring and students who did not.

In [ ]:
overall = (
    students.groupby("tutoring")["passed"]
    .agg(["count", "sum", "mean"])
    .rename(columns={"count": "Students", "sum": "Passed", "mean": "Pass rate"})
)
overall.style.format({"Pass rate": "{:.1%}"})

In [ ]:
plt.figure(figsize=(7, 5))
plt.bar(overall.index, overall["Pass rate"])
plt.ylim(0, 1)
plt.xlabel("Tutoring participation")
plt.ylabel("Pass rate")
plt.title("Overall pass rate by tutoring participation")
plt.grid(axis="y", alpha=0.3)
plt.show()

## Stop and interpret

1. Which group had the higher overall pass rate?
2. What causal conclusion might someone make?
3. Does this result prove tutoring caused lower performance?
4. What variable might influence both tutoring participation and passing?

### My interpretation

**Observed association:**  

**Tempting causal conclusion:**  

**Why that conclusion may be unsafe:**  

**Possible confounder:**

# Part 4 — Examine Who Attended Tutoring

Are the tutoring and non-tutoring groups comparable before tutoring occurred?

In [ ]:
participation_counts = pd.crosstab(students["preparation"], students["tutoring"])
participation_rates = pd.crosstab(
    students["preparation"], students["tutoring"], normalize="index"
)

display(participation_counts)
display(participation_rates.style.format("{:.1%}"))

# Part 5 — Stratify by Prior Preparation

Calculate pass rates separately for high-preparation and low-preparation students.

In [ ]:
stratified = (
    students.groupby(["preparation", "tutoring"])["passed"]
    .agg(["count", "sum", "mean"])
    .rename(columns={"count": "Students", "sum": "Passed", "mean": "Pass rate"})
    .reset_index()
)
stratified.style.format({"Pass rate": "{:.1%}"})

In [ ]:
pivot_rates = students.pivot_table(
    index="preparation",
    columns="tutoring",
    values="passed",
    aggfunc="mean",
)
pivot_rates.style.format("{:.1%}")

In [ ]:
pivot_rates[["No", "Yes"]].plot(kind="bar", figsize=(8, 5))
plt.ylim(0, 1)
plt.xlabel("Prior preparation")
plt.ylabel("Pass rate")
plt.title("Pass rates within each preparation group")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.legend(title="Tutoring")
plt.show()

In [ ]:
within_group_effect = pivot_rates.copy()
within_group_effect["Difference: Yes - No"] = (
    within_group_effect["Yes"] - within_group_effect["No"]
)
within_group_effect.style.format("{:.1%}")

## Explain the reversal

1. Within the high-preparation group, which students passed at a higher rate?
2. Within the low-preparation group, which students passed at a higher rate?
3. Why did the overall comparison point in the opposite direction?

<details>
<summary><strong>Reveal a possible explanation</strong></summary>

Within both preparation groups, students who attended tutoring had a higher pass rate. However, most tutoring participants came from the low-preparation group, which had a lower baseline probability of passing. Most nonparticipants came from the high-preparation group, which had a higher baseline probability of passing.

When the groups were combined, the different group composition overwhelmed the within-group advantage associated with tutoring. This reversal is an example of **Simpson's paradox**.

</details>

# Part 6 — Understand the Causal Structure

A confounder is associated with both the exposure and the outcome.

In this example:

- Prior preparation influences tutoring participation.
- Prior preparation influences passing.
- Tutoring may also influence passing.

```text
Prior preparation ─────► Tutoring participation
        │
        └──────────────► Passing

Tutoring participation ─► Passing
```

The statements below are not equivalent:

- “Tutoring participants had a lower overall pass rate.”
- “Tutoring caused students to pass at a lower rate.”

The first is descriptive. The second is causal and requires stronger evidence.

# Part 7 — Calculate a Standardized Comparison

Give the high- and low-preparation groups equal weight. This does not prove causation, but it reduces distortion caused by the different group composition.

In [ ]:
equal_weight_adjusted = pd.Series({
    "No tutoring": pivot_rates["No"].mean(),
    "Tutoring": pivot_rates["Yes"].mean(),
})

display(equal_weight_adjusted.to_frame("Equally weighted pass rate").style.format("{:.1%}"))

adjusted_difference = (
    equal_weight_adjusted["Tutoring"] - equal_weight_adjusted["No tutoring"]
)
print(f"Equally weighted difference: {adjusted_difference:.1%}")

### Interpret the adjusted result

**Why the result changed:**  

**What the adjusted comparison suggests:**  

**What it still does not prove:**

# Part 8 — Use AI as a Skeptical Reviewer

Use AI only after completing your own analysis. Do not ask, “Does tutoring work?”

Choose one prompt:

> Our overall data suggest tutoring students perform worse, but within each preparation group they perform better. Ask us four questions that test whether our explanation is complete.

> Identify three alternative explanations for the observed pattern besides a true tutoring effect.

> Act as a skeptical statistician. What assumptions are we making when we adjust only for preparation level?

> What additional variables might influence both tutoring participation and passing?

> Explain what evidence would be needed before making a causal claim about tutoring.

### AI-response evaluation

**Prompt used:**  

**One useful point:**  

**One claim requiring verification:**  

**One missing variable suggested by AI:**  

**One unsupported assumption:**  

**One idea I will use:**  

**One idea I will reject and why:**

# Part 9 — What Other Biases Could Remain?

Even after controlling for preparation, the groups may differ in:

- motivation;
- attendance;
- available study time;
- employment hours;
- instructor;
- course difficulty;
- prior GPA;
- internet access;
- tutoring intensity;
- whether students completed the tutoring program.

Discuss:

1. Which variables should be measured?
2. Which variables occurred before tutoring?
3. Which may be consequences of tutoring?
4. Why could controlling for the wrong variable create additional bias?

# Part 10 — Design a Stronger Study

Choose one or combine several approaches:

- randomized encouragement design;
- matched comparison;
- regression adjustment;
- before-and-after comparison with a comparison group;
- randomized controlled trial.

## Study-design proposal

**Selected design:**  

**Population:**  

**Intervention or exposure:**  

**Comparison group:**  

**Outcome:**  

**Variables to collect:**  

**How confounding will be reduced:**  

**Ethical concerns:**  

**Remaining limitations:**

# Part 11 — Communicate the Result Responsibly

Rewrite this headline:

> “Tutoring Makes Students Less Likely to Pass”

Your revision should describe the association accurately, avoid an unsupported causal claim, mention the preparation imbalance, and explain why further analysis is needed.

### Revised headline

**Headline:**  

**Explanation:**  

<details>
<summary><strong>Reveal one responsible version</strong></summary>

**Headline:** Students Who Used Tutoring Had a Lower Overall Pass Rate, but Preparation Differences Changed the Picture

**Explanation:** Tutoring participants were much more likely to come from the low-preparation group. Within both preparation groups, students who attended tutoring had higher pass rates than comparable nonparticipants. Because tutoring was optional and students were not randomly assigned, the analysis does not by itself prove that tutoring caused the difference.

</details>

# Part 12 — Transfer the Reasoning

Complete this section without AI.

A company evaluates an optional professional-development program. Overall, employees who joined received fewer promotions than employees who did not. However, employees with lower initial performance ratings were much more likely to enroll.

1. What is the exposure?
2. What is the outcome?
3. What is the likely confounder?
4. What comparison should be performed before drawing a conclusion?
5. What causal claim would be unsafe?
6. What evidence would strengthen the analysis?

### Transfer response

**Exposure:**  

**Outcome:**  

**Likely confounder:**  

**Required comparison:**  

**Unsafe causal claim:**  

**Additional evidence needed:**

# Part 13 — Exit Reflection

Complete the following:

- Before this lesson, I thought correlation meant …
- The aggregate result was misleading because …
- The confounding variable was …
- Simpson's paradox occurred because …
- AI helped me examine …
- One claim I would no longer make is …
- One question I will ask before making a causal claim is …

# Instructor Review Checklist

- [ ] Students stated an initial conclusion before seeing the full analysis.
- [ ] Students separated association from causal interpretation.
- [ ] Students checked data quality before analysis.
- [ ] Students examined who entered each comparison group.
- [ ] Students calculated aggregate and subgroup rates.
- [ ] Students explained Simpson's paradox in plain language.
- [ ] Students identified the confounder's connection to exposure and outcome.
- [ ] AI challenged reasoning rather than supplied the conclusion.
- [ ] Students evaluated AI output critically.
- [ ] Students proposed a stronger study design.
- [ ] Students rewrote a misleading causal headline.
- [ ] Students transferred the reasoning to a new context.
- [ ] Reflection made changes in thinking visible.

# Closing Principle

Data analysis does not end when we calculate a percentage or fit a model.

A responsible data scientist asks:

- Who entered each group?
- What happened before the exposure?
- Which variables influence both the exposure and the outcome?
- Could aggregation be hiding important differences?
- What evidence supports a causal conclusion?
- What uncertainty remains?

The strongest analyst is not the person who finds a pattern fastest. It is the person who knows when that pattern may be telling an incomplete story.